# 1. Setup & Environment Configuration
Install required external packages, configure random seeds for reproducible experiments, and detect computational hardware.

In [13]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


Using device: cuda


# 2. Ontology Knowledge Graph Engine
Defines the `OntologyEngine` to extract lexical emotion signals, appraisal values, polarity, and intensity from an RDF Knowledge Graph.

In [14]:
# Initialize OntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsmec.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = OntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.85),
    manual_boost=onto_cfg.get('manual_boost', 1.2),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.5),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


Found ontology file: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Loading Knowledge Graph: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Successfully loaded 90420 triples.
Lexicon initialized with 2540 entries.


# 3. Ontology Diagnostics & Validation Suite
Validates the ontology feature extraction logic across key test cases (basic sentiments, soft negation, scope handling, and ambiguity filtering).

In [15]:
def decode_vector(vec, engine):
    """Decode a 24-dimensional feature vector into human-readable representations."""
    emo_vals = vec[:7]
    if np.max(emo_vals) > 0.01:
        top_idx = np.argmax(emo_vals)
        emotion = engine.ALLEMOTIONS[top_idx]
        score = emo_vals[top_idx]
    else:
        emotion = "Neutral/None"
        score = 0.0

    pol_vals = vec[19:22]
    pol_labels = ["Positive", "Negative", "Neutral"]
    if np.max(pol_vals) > 0:
        polarity = pol_labels[np.argmax(pol_vals)]
    else:
        polarity = "Unknown"

    has_neg = vec[22] > 0
    return emotion, score, polarity, has_neg


# Diagnostic test suite
test_cases = [
    ("món này ngon tuyệt", "Happiness", "Positive", "Basic Positive"),
    ("thất vọng tràn trề", "Sadness", "Negative", "Basic Negative"),
    (
        "tôi không thích nó",
        "Neutral/None",
        "Negative",
        "Soft Negation (Không thích)",
    ),
    ("chẳng vui chút nào", "Neutral/None", "Negative", "Soft Negation (Chẳng vui)"),
    (
        "tôi không hề thích",
        "Neutral/None",
        "Negative",
        "Scope Handling (Không hề...)",
    ),
    (
        "không phải là ghét",
        "Neutral/None",
        "Positive",
        "Double Negation (Không ghét)",
    ),
    ("làm hay nghỉ", "Neutral/None", "Unknown", "Disambiguation: Hay (Or)"),
    ("bún chả rất ngon", "Happiness", "Positive", "Disambiguation: Chả (Food)"),
    (
        "tôi là sinh viên",
        "Neutral/None",
        "Unknown",
        "Disambiguation: Pronouns/Stopwords",
    ),
]

print("==================== ONTOLOGY DIAGNOSTIC REPORT ====================")
results = []
pass_count = 0

for text, exp_emo, exp_pol, desc in test_cases:
    vec, _, trace = ontengine.getvector(text, debug=True)
    out_emo, out_score, out_pol, out_neg = decode_vector(vec, ontengine)

    emo_pass = (
        (out_score < 0.5 or out_emo == "Neutral")
        if exp_emo == "Neutral/None"
        else (out_emo == exp_emo and out_score > 0.3)
    )
    pol_pass = True if exp_pol == "Unknown" else (out_pol == exp_pol)

    status = "FAIL"
    if "Disambiguation: Chả" in desc and out_neg:
        status = "FAIL (False Negation)"
    elif "Soft Negation" in desc and out_pol == "Positive":
        status = "FAIL (Wrong Polarity)"
    elif emo_pass and pol_pass:
        status = "PASS"
        pass_count += 1

    matched_str = ", ".join([t["phrase"] for t in trace])
    results.append(
        {
            "Scenario": desc,
            "Input": text,
            "Matched": matched_str,
            "Prediction": f"{out_emo} ({out_score:.2f}) | {out_pol}",
            "NegFlag": "YES" if out_neg else "NO",
            "Status": status,
        }
    )

df_res = pd.DataFrame(results)
print(
    df_res[["Input", "Matched", "Prediction", "Status"]].to_string(index=False)
)
print("-" * 68)
print(f"Summary: {pass_count}/{len(test_cases)} test cases passed.")
print("====================================================================")

==================== ONTOLOGY DIAGNOSTIC REPORT ====================
             Input           Matched                    Prediction                Status
món này ngon tuyệt             tuyệt   Happiness (0.75) | Positive                  PASS
thất vọng tràn trề         thất vọng     Sadness (0.58) | Negative                  PASS
tôi không thích nó       không thích     Disgust (0.55) | Negative                  FAIL
chẳng vui chút nào               vui   Happiness (0.79) | Positive FAIL (Wrong Polarity)
tôi không hề thích      không, thích   Happiness (0.45) | Positive                  FAIL
không phải là ghét không, phải, ghét       Anger (0.17) | Negative                  FAIL
      làm hay nghỉ              nghỉ     Sadness (0.39) | Negative                  PASS
  bún chả rất ngon               rất   Happiness (0.42) | Positive                  PASS
  tôi là sinh viên                   Neutral/None (0.00) | Unknown                  PASS
-----------------------------------------

# 4. Dataset Preprocessing & DataLoaders
Loads the UIT-VSMEC dataset, encodes labels, extracts ontology vectors per sentence, and prepares PyTorch DataLoaders.

In [16]:
# 1. Trích xuất vector cảm xúc thô VnEmoLex (Doãn & Lưu, 2022)
# Chỉ cộng dồn điểm cảm xúc của từ vựng, không sử dụng negation, appraisal, polarity hay ontology similarity
def get_vnemolex_vector(text, engine):
    tokens = ViTokenizer.tokenize(str(text).lower()).split()
    flat_tokens = [engine._norm(t) for t in tokens if engine._norm(t)]
    v_emo = np.zeros(len(engine.ALLEMOTIONS), dtype=np.float32)

    idx, n = 0, len(flat_tokens)
    while idx < n:
        matched_entry, matched_len = None, 0
        for l_span in range(min(engine.mwe_max_len, n - idx), 0, -1):
            phrase = " ".join(flat_tokens[idx : idx + l_span])
            if phrase in engine.lexicon_map:
                matched_entry = engine.lexicon_map[phrase]
                matched_len = l_span
                break

        if not matched_entry:
            idx += 1
            continue

        for ent in matched_entry:
            if ent.get("isnegation", False):
                continue
            score = ent.get("score", 1.0)
            for e, w in ent.get("emotions", {}).items():
                if e in engine.ALLEMOTIONS:
                    v_emo[engine.ALLEMOTIONS.index(e)] += score * w

        idx += matched_len

    return v_emo  # Vector điểm cảm xúc thô 7 chiều


# 2. Khởi tạo Tokenizer cho cả 2 backbone
tokenizer_phobert = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
tokenizer_visobert = AutoTokenizer.from_pretrained("uitnlp/visobert")


class DualBackboneDataset(Dataset):

    def __init__(self, df, tokenizer, engine, is_phobert=False, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.engine = engine
        self.is_phobert = is_phobert
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        txt = str(self.df.iloc[idx]["Sentence"])
        lbl = self.df.iloc[idx]["label_idx"]

        # PhoBERT yêu cầu tách từ (segmentation) trước khi đưa vào BPE
        input_text = ViTokenizer.tokenize(txt) if self.is_phobert else txt

        enc = self.tokenizer(
            input_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        vnemolex_vec = get_vnemolex_vector(txt, self.engine)
        ont_vec = self.engine.getvector(txt)
        if isinstance(ont_vec, tuple):
            ont_vec = ont_vec[0]

        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "lexicon_features": torch.tensor(vnemolex_vec, dtype=torch.float),
            "ontology_features": torch.tensor(ont_vec, dtype=torch.float),
            "labels": torch.tensor(lbl, dtype=torch.long),
        }


# 3. Load dữ liệu UIT-VSMEC
BASE_DATA_PATH = "../../data/vsmec"

df_train = pd.read_excel(f"{BASE_DATA_PATH}/train_nor_811.xlsx")
df_valid = pd.read_excel(f"{BASE_DATA_PATH}/valid_nor_811.xlsx")
df_test = pd.read_excel(f"{BASE_DATA_PATH}/test_nor_811.xlsx")

for df in [df_train, df_valid, df_test]:
    df.dropna(subset=["Sentence", "Emotion"], inplace=True)

labels = sorted(df_train["Emotion"].unique().astype(str))
le = LabelEncoder()
le.fit(labels)
print(f"Classes ({len(le.classes_)}): {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df["label_idx"] = le.transform(df["Emotion"].astype(str))

# Dataloaders riêng cho PhoBERT
BATCH_SIZE = 32
train_loader_pho = DataLoader(
    DualBackboneDataset(
        df_train, tokenizer_phobert, ontengine, is_phobert=True
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
valid_loader_pho = DataLoader(
    DualBackboneDataset(
        df_valid, tokenizer_phobert, ontengine, is_phobert=True
    ),
    batch_size=BATCH_SIZE,
)
test_loader_pho = DataLoader(
    DualBackboneDataset(df_test, tokenizer_phobert, ontengine, is_phobert=True),
    batch_size=BATCH_SIZE,
)

# Dataloaders riêng cho ViSoBERT
train_loader_viso = DataLoader(
    DualBackboneDataset(
        df_train, tokenizer_visobert, ontengine, is_phobert=False
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
valid_loader_viso = DataLoader(
    DualBackboneDataset(
        df_valid, tokenizer_visobert, ontengine, is_phobert=False
    ),
    batch_size=BATCH_SIZE,
)
test_loader_viso = DataLoader(
    DualBackboneDataset(
        df_test, tokenizer_visobert, ontengine, is_phobert=False
    ),
    batch_size=BATCH_SIZE,
)

cls_w = compute_class_weight(
    "balanced",
    classes=np.unique(df_train["label_idx"]),
    y=df_train["label_idx"],
)
weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
print(f"Computed Class Weights: {weights_tensor}")

Classes (7): ['Anger' 'Disgust' 'Enjoyment' 'Fear' 'Other' 'Sadness' 'Surprise']
Computed Class Weights: tensor([2.0270, 0.7400, 0.5087, 2.4924, 0.7763, 0.8369, 3.2751],
       device='cuda:0')


# 5. Model Architectures
Defines neural-symbolic fusion architectures:
1. `ViSoBERT_Fusion`: Supports baseline, raw concatenation, and gated projection.
2. `ViSoBERT_DeepOntology`: Multi-layer bottleneck projection with residual gating.
3. `ViSoBERT_Ontology_CrossAttention`: Multi-head cross-attention querying discrete ontology concept tokens.

In [17]:
# Import lexicon vs ontology comparative models from src.models
from src.models.lexicon_ontology_models import (
    Model_Baseline,
    Model_VnEmoLex_Concat,
    Model_Ontology_RawGate,
)

VNEMOLEX_DIM = 7
ONTOLOGY_DIM = 24
print('✅ Models loaded from src.models.lexicon_ontology_models: Model_Baseline, Model_VnEmoLex_Concat, Model_Ontology_RawGate')


# 6. Training & Evaluation Pipeline
Contains the training loop with early stopping, model checkpointing, and evaluation utilities.

In [18]:
def evaluate(model, loader, feature_key="lexicon_features"):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = batch[feature_key].to(device) if feature_key in batch else None

            outputs = model(ids, mask, feat)
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return accuracy_score(targets, preds), f1_score(
        targets, preds, average="macro"
    )


def train_experiment(
    model,
    name,
    train_loader,
    valid_loader,
    feature_key="lexicon_features",
    epochs=15,
    patience=4,
):
    print(f"\nTraining Model: {name}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=2e-5, weight_decay=0.01
    )
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    best_f1 = 0.0
    patience_counter = 0
    best_weights = copy.deepcopy(
        model.module.state_dict()
        if isinstance(model, nn.DataParallel)
        else model.state_dict()
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = (
                batch[feature_key].to(device) if feature_key in batch else None
            )

            outputs = model(ids, mask, feat)
            loss = criterion(outputs, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc, val_f1 = evaluate(model, valid_loader, feature_key=feature_key)
        print(
            f"Epoch {epoch + 1:02d} | Train Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}",
            end=" ",
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            print("(New best score)")
        else:
            patience_counter += 1
            print(f"(Patience: {patience_counter}/{patience})")

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_weights)
    else:
        model.load_state_dict(best_weights)
    return model

# 7. Model Training & Experimentation
Trains multiple model configurations:
- Baseline (ViSoBERT only)
- Raw Concat & Raw Gate
- Dense (64d) Concat & Dense (64d) Gate
- Residual Bottleneck Gate
- The ontology-guided cross-attention model.

In [19]:
NUM_CLASSES = len(le.classes_)
PHO_ID = "vinai/phobert-base-v2"
VISO_ID = "uitnlp/visobert"

# ==================== NHÓM 1: PHOBERT ====================
# 1. PhoBERT Baseline
m_pho_base = Model_Baseline(NUM_CLASSES, model_name=PHO_ID)
m_pho_base = train_experiment(
    m_pho_base,
    "1. PhoBERT Baseline",
    train_loader_pho,
    valid_loader_pho,
    feature_key="lexicon_features",
)

# 2. PhoBERT + VnEmoLex Concat (Doãn & Lưu, 2022)
m_pho_lex = Model_VnEmoLex_Concat(NUM_CLASSES, model_name=PHO_ID)
m_pho_lex = train_experiment(
    m_pho_lex,
    "2. PhoBERT + VnEmoLex (Doãn & Lưu, 2022)",
    train_loader_pho,
    valid_loader_pho,
    feature_key="lexicon_features",
)

# 3. PhoBERT + Full Ontology (RawGate)
m_pho_ont = Model_Ontology_RawGate(NUM_CLASSES, model_name=PHO_ID)
m_pho_ont = train_experiment(
    m_pho_ont,
    "3. PhoBERT + Full Ontology (RawGate)",
    train_loader_pho,
    valid_loader_pho,
    feature_key="ontology_features",
)

# ==================== NHÓM 2: VISOBERT ====================
# 4. ViSoBERT Baseline
m_viso_base = Model_Baseline(NUM_CLASSES, model_name=VISO_ID)
m_viso_base = train_experiment(
    m_viso_base,
    "4. ViSoBERT Baseline",
    train_loader_viso,
    valid_loader_viso,
    feature_key="lexicon_features",
)

# 5. ViSoBERT + VnEmoLex Concat (Doãn & Lưu, 2022)
m_viso_lex = Model_VnEmoLex_Concat(NUM_CLASSES, model_name=VISO_ID)
m_viso_lex = train_experiment(
    m_viso_lex,
    "5. ViSoBERT + VnEmoLex (Doãn & Lưu, 2022)",
    train_loader_viso,
    valid_loader_viso,
    feature_key="lexicon_features",
)

# 6. ViSoBERT + Full Ontology (RawGate)
m_viso_ont = Model_Ontology_RawGate(NUM_CLASSES, model_name=VISO_ID)
m_viso_ont = train_experiment(
    m_viso_ont,
    "6. ViSoBERT + Full Ontology (RawGate)",
    train_loader_viso,
    valid_loader_viso,
    feature_key="ontology_features",
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 1. PhoBERT Baseline
Epoch 01 | Train Loss: 1.6676 | Val F1: 0.4488 | Val Acc: 0.4883 (New best score)
Epoch 02 | Train Loss: 1.1214 | Val F1: 0.5540 | Val Acc: 0.5991 (New best score)
Epoch 03 | Train Loss: 0.8791 | Val F1: 0.5476 | Val Acc: 0.5889 (Patience: 1/4)
Epoch 04 | Train Loss: 0.6763 | Val F1: 0.5482 | Val Acc: 0.5787 (Patience: 2/4)
Epoch 05 | Train Loss: 0.5147 | Val F1: 0.5717 | Val Acc: 0.6224 (New best score)
Epoch 06 | Train Loss: 0.3989 | Val F1: 0.5521 | Val Acc: 0.5962 (Patience: 1/4)
Epoch 07 | Train Loss: 0.3041 | Val F1: 0.5513 | Val Acc: 0.6050 (Patience: 2/4)
Epoch 08 | Train Loss: 0.2297 | Val F1: 0.5958 | Val Acc: 0.6429 (New best score)
Epoch 09 | Train Loss: 0.2026 | Val F1: 0.5938 | Val Acc: 0.6341 (Patience: 1/4)
Epoch 10 | Train Loss: 0.1484 | Val F1: 0.5601 | Val Acc: 0.6064 (Patience: 2/4)
Epoch 11 | Train Loss: 0.1184 | Val F1: 0.5666 | Val Acc: 0.6137 (Patience: 3/4)
Epoch 12 | Train Loss: 0.1073 | Val F1: 0.5881 | Val Acc: 0.6239 (Pa

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 5. ViSoBERT + VnEmoLex (Doãn & Lưu, 2022)
Epoch 01 | Train Loss: 1.5626 | Val F1: 0.4908 | Val Acc: 0.5277 (New best score)
Epoch 02 | Train Loss: 1.0167 | Val F1: 0.5257 | Val Acc: 0.5773 (New best score)
Epoch 03 | Train Loss: 0.6789 | Val F1: 0.5270 | Val Acc: 0.5743 (New best score)
Epoch 04 | Train Loss: 0.4000 | Val F1: 0.5130 | Val Acc: 0.5481 (Patience: 1/4)
Epoch 05 | Train Loss: 0.2244 | Val F1: 0.5287 | Val Acc: 0.5714 (New best score)
Epoch 06 | Train Loss: 0.1323 | Val F1: 0.5253 | Val Acc: 0.5787 (Patience: 1/4)
Epoch 07 | Train Loss: 0.0831 | Val F1: 0.5341 | Val Acc: 0.5875 (New best score)
Epoch 08 | Train Loss: 0.0618 | Val F1: 0.5103 | Val Acc: 0.5569 (Patience: 1/4)
Epoch 09 | Train Loss: 0.0515 | Val F1: 0.5353 | Val Acc: 0.5904 (New best score)
Epoch 10 | Train Loss: 0.0358 | Val F1: 0.5316 | Val Acc: 0.5875 (Patience: 1/4)
Epoch 11 | Train Loss: 0.0496 | Val F1: 0.5414 | Val Acc: 0.5918 (New best score)
Epoch 12 | Train Loss: 0.0263 | Val F1: 0.5

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 6. ViSoBERT + Full Ontology (RawGate)
Epoch 01 | Train Loss: 1.5361 | Val F1: 0.5031 | Val Acc: 0.5394 (New best score)
Epoch 02 | Train Loss: 0.9890 | Val F1: 0.5475 | Val Acc: 0.5743 (New best score)
Epoch 03 | Train Loss: 0.6644 | Val F1: 0.5248 | Val Acc: 0.5773 (Patience: 1/4)
Epoch 04 | Train Loss: 0.4063 | Val F1: 0.5293 | Val Acc: 0.5758 (Patience: 2/4)
Epoch 05 | Train Loss: 0.2230 | Val F1: 0.5421 | Val Acc: 0.5831 (Patience: 3/4)
Epoch 06 | Train Loss: 0.1290 | Val F1: 0.5382 | Val Acc: 0.5918 (Patience: 4/4)
Early stopping triggered.


# 8. Test Set Evaluation & Benchmark Comparison
Evaluates all trained variants on the test set and prints a comprehensive comparison table and classification metrics.

In [20]:
def get_predictions(model, loader, feature_key="lexicon_features"):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = (
                batch[feature_key].to(device) if feature_key in batch else None
            )

            outputs = model(ids, mask, feat)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return targets, preds


# Thu thập dự đoán PhoBERT
y_true, p_pho_base = get_predictions(
    m_pho_base, test_loader_pho, feature_key="lexicon_features"
)
_, p_pho_lex = get_predictions(
    m_pho_lex, test_loader_pho, feature_key="lexicon_features"
)
_, p_pho_ont = get_predictions(
    m_pho_ont, test_loader_pho, feature_key="ontology_features"
)

# Thu thập dự đoán ViSoBERT
_, p_viso_base = get_predictions(
    m_viso_base, test_loader_viso, feature_key="lexicon_features"
)
_, p_viso_lex = get_predictions(
    m_viso_lex, test_loader_viso, feature_key="lexicon_features"
)
_, p_viso_ont = get_predictions(
    m_viso_ont, test_loader_viso, feature_key="ontology_features"
)

models_preds = [
    ("PhoBERT", "Baseline (No External)", p_pho_base),
    ("PhoBERT", "VnEmoLex Concat (Doãn & Lưu, 2022)", p_pho_lex),
    ("PhoBERT", "Full Ontology RawGate (Ours)", p_pho_ont),
    ("ViSoBERT", "Baseline (No External)", p_viso_base),
    ("ViSoBERT", "VnEmoLex Concat (Doãn & Lưu, 2022)", p_viso_lex),
    ("ViSoBERT", "Full Ontology RawGate (Ours)", p_viso_ont),
]

summary_records = []
for backbone, method, preds in models_preds:
    summary_records.append(
        {
            "Backbone": backbone,
            "Knowledge Integration": method,
            "Accuracy": accuracy_score(y_true, preds),
            "Macro F1": f1_score(y_true, preds, average="macro"),
        }
    )

print("\n================ FINAL TEST BENCHMARK RESULTS ================")
df_benchmark = pd.DataFrame(summary_records)
print(df_benchmark.round(4).to_string(index=False))

print(
    "\n========== REPORT: PhoBERT + VnEmoLex (Doãn & Lưu, 2022) =========="
)
print(
    classification_report(y_true, p_pho_lex, target_names=le.classes_, digits=4)
)

print(
    "\n========== REPORT: ViSoBERT + VnEmoLex (Doãn & Lưu, 2022) =========="
)
print(
    classification_report(
        y_true, p_viso_lex, target_names=le.classes_, digits=4
    )
)


================ FINAL TEST BENCHMARK RESULTS ================
Backbone              Knowledge Integration  Accuracy  Macro F1
 PhoBERT             Baseline (No External)    0.6710    0.6513
 PhoBERT VnEmoLex Concat (Doãn & Lưu, 2022)    0.6681    0.6528
 PhoBERT       Full Ontology RawGate (Ours)    0.6551    0.6326
ViSoBERT             Baseline (No External)    0.5887    0.5713
ViSoBERT VnEmoLex Concat (Doãn & Lưu, 2022)    0.6061    0.5828
ViSoBERT       Full Ontology RawGate (Ours)    0.6003    0.5839

========== REPORT: PhoBERT + VnEmoLex (Doãn & Lưu, 2022) ==========
              precision    recall  f1-score   support

       Anger     0.4737    0.6750    0.5567        40
     Disgust     0.6612    0.6061    0.6324       132
   Enjoyment     0.7282    0.7772    0.7519       193
        Fear     0.6889    0.6739    0.6813        46
       Other     0.6204    0.5194    0.5654       129
     Sadness     0.6885    0.7241    0.7059       116
    Surprise     0.7059    0.6486    0.6

# 9. Neuro-Symbolic Rule-Based Reasoner
Applies post-processing domain rules from the ontology to override low-confidence neural model predictions.

In [21]:
class NeuroSymbolicReasonerVSMEC:
    """Post-processing override module combining neural predictions with ontology rules."""

    def __init__(
        self,
        model,
        engine,
        tokenizer,
        device,
        ontology_threshold=0.4,
        model_conf_limit=0.65,
        is_phobert=False,
    ):
        self.model = (
            model.module if isinstance(model, nn.DataParallel) else model
        )
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.ont_threshold = ontology_threshold
        self.model_conf_limit = model_conf_limit
        self.is_phobert = is_phobert
        self.idx2label = {i: label for i, label in enumerate(le.classes_)}

    def infer(self, text):
        input_text = ViTokenizer.tokenize(text) if self.is_phobert else text
        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        vec, _, _ = self.engine.getvector(text, debug=True)
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(
            self.device
        )

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)

        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        neural_label = self.idx2label[pred_idx]

        # Symbolic override logic
        ont_emotions = vec[0:7]
        max_ont_idx = np.argmax(ont_emotions)
        max_ont_score = ont_emotions[max_ont_idx]
        ont_label_name = self.engine.ALLEMOTIONS[max_ont_idx]

        ont_to_vsmec = {"Happiness": "Enjoyment", "Neutral": "Other"}
        final_ont_label = ont_to_vsmec.get(ont_label_name, ont_label_name)
        final_label = neural_label

        if neural_label == "Other" or confidence < self.model_conf_limit:
            if (
                max_ont_score > self.ont_threshold
                and final_ont_label in le.classes_
            ):
                final_label = final_ont_label

        return neural_label, final_label


def run_rule_evaluation(model, tokenizer, is_phobert=False, name="ViSoBERT"):
    reasoner = NeuroSymbolicReasonerVSMEC(
        model=model,
        engine=ontengine,
        tokenizer=tokenizer,
        device=device,
        ontology_threshold=0.4,
        model_conf_limit=0.65,
        is_phobert=is_phobert,
    )

    y_true_ns = []
    y_pred_base = []
    y_pred_neuro = []

    for _, row in df_test.iterrows():
        text = row["Sentence"]
        true_label_str = row["Emotion"]

        base_lbl, neuro_lbl = reasoner.infer(text)
        y_true_ns.append(true_label_str)
        y_pred_base.append(base_lbl)
        y_pred_neuro.append(neuro_lbl)

    acc_base = accuracy_score(y_true_ns, y_pred_base)
    f1_base = f1_score(y_true_ns, y_pred_base, average="macro")
    acc_neuro = accuracy_score(y_true_ns, y_pred_neuro)
    f1_neuro = f1_score(y_true_ns, y_pred_neuro, average="macro")

    overridden_count = sum(
        1 for b, n in zip(y_pred_base, y_pred_neuro) if b != n
    )
    print(
        f"[{name}] Total overridden samples: {overridden_count}/{len(df_test)} ({overridden_count / len(df_test) * 100:.2f}%)"
    )

    return {
        "Model": name,
        "Baseline Acc": acc_base,
        "Rule Acc": acc_neuro,
        "Delta Acc": acc_neuro - acc_base,
        "Baseline F1": f1_base,
        "Rule F1": f1_neuro,
        "Delta F1": f1_neuro - f1_base,
    }


records = []
records.append(
    run_rule_evaluation(
        m_pho_base, tokenizer_phobert, is_phobert=True, name="PhoBERT"
    )
)
records.append(
    run_rule_evaluation(
        m_viso_base, tokenizer_visobert, is_phobert=False, name="ViSoBERT"
    )
)

df_rule_res = pd.DataFrame(records)
print("\n================ RULE-BASED POST-PROCESSING EVALUATION ================")
print(df_rule_res.round(4).to_string(index=False))

[PhoBERT] Total overridden samples: 46/693 (6.64%)
[ViSoBERT] Total overridden samples: 91/693 (13.13%)

================ RULE-BASED POST-PROCESSING EVALUATION ================
   Model  Baseline Acc  Rule Acc  Delta Acc  Baseline F1  Rule F1  Delta F1
 PhoBERT        0.6710    0.6551    -0.0159       0.6513   0.6228   -0.0285
ViSoBERT        0.5887    0.5700    -0.0188       0.5713   0.5409   -0.0304


# 10. Statistical Significance Testing (Paired Permutation Test)
Performs non-parametric paired permutation tests (10,000 iterations) to determine if improvements are statistically significant ($p < 0.05$).

In [22]:
from src.significance_test import paired_permutation_test, macro_f1


comparisons = [
    # PhoBERT
    (
        "PhoBERT: Baseline vs VnEmoLex (Doãn & Lưu)",
        p_pho_base,
        p_pho_lex,
    ),
    (
        "PhoBERT: VnEmoLex vs Full Ontology",
        p_pho_lex,
        p_pho_ont,
    ),
    (
        "PhoBERT: Baseline vs Full Ontology",
        p_pho_base,
        p_pho_ont,
    ),
    # ViSoBERT
    (
        "ViSoBERT: Baseline vs VnEmoLex (Doãn & Lưu)",
        p_viso_base,
        p_viso_lex,
    ),
    (
        "ViSoBERT: VnEmoLex vs Full Ontology",
        p_viso_lex,
        p_viso_ont,
    ),
    (
        "ViSoBERT: Baseline vs Full Ontology",
        p_viso_base,
        p_viso_ont,
    ),
]

print(
    "\n============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) =============="
)

results = []
for name, preds_a, preds_b in comparisons:
    f1_a, f1_b, diff_f1, p_val_f1 = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=macro_f1
    )
    acc_a, acc_b, diff_acc, p_val_acc = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=accuracy_score
    )

    sig_f1 = "Significant (p < 0.05)" if p_val_f1 < 0.05 else "Not Significant"
    sig_acc = (
        "Significant (p < 0.05)" if p_val_acc < 0.05 else "Not Significant"
    )

    results.append(
        {
            "Comparison": name,
            "F1 (Model A)": round(f1_a, 4),
            "F1 (Model B)": round(f1_b, 4),
            "Diff (F1)": round(diff_f1, 4),
            "p-val (F1)": round(p_val_f1, 5),
            "Sig (F1)": sig_f1,
            "p-val (Acc)": round(p_val_acc, 5),
            "Sig (Acc)": sig_acc,
        }
    )

df_perm = pd.DataFrame(results)
print(df_perm.to_string(index=False))
print(
    "\n================================================================================"
)


============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) ==============
                                 Comparison  F1 (Model A)  F1 (Model B)  Diff (F1)  p-val (F1)        Sig (F1)  p-val (Acc)       Sig (Acc)
 PhoBERT: Baseline vs VnEmoLex (Doãn & Lưu)        0.6513        0.6528     0.0015     0.93261 Not Significant      0.92551 Not Significant
         PhoBERT: VnEmoLex vs Full Ontology        0.6528        0.6326     0.0202     0.25187 Not Significant      0.43206 Not Significant
         PhoBERT: Baseline vs Full Ontology        0.6513        0.6326     0.0187     0.29227 Not Significant      0.32957 Not Significant
ViSoBERT: Baseline vs VnEmoLex (Doãn & Lưu)        0.5713        0.5828     0.0115     0.56914 Not Significant      0.35306 Not Significant
        ViSoBERT: VnEmoLex vs Full Ontology        0.5828        0.5839     0.0011     0.95490 Not Significant      0.79902 Not Significant
        ViSoBERT: Baseline vs Full Ontology        0.5713        0.5839     0.